# Мы приступаем с вами к выполнению заключительной лабораторной работы , которая связана с Flask . В данной лаборатоной работе , мы будем с вами говорить , как же добавить к вашему сайду базу данных(не txt файл) , как реализуется регистрация и вход в свою личную учетную запись . 
<br>

<br> 

# Самое первое и самое главное , что нам нужно сделать , так это поменять app.py . Сейчас наш app.py выглядит вот так , по сути у нас подключены только html страницы 

In [ ]:
from flask import Flask, render_template

app = Flask(__name__)

@app.route('/')
def button_page():
    return render_template('button.html')

@app.route('/register')
def registration_page():
    return render_template('index_registration.html')

if __name__ == '__main__':
    app.run(debug=True)

# Теперь допишим наш код 

Сначала установим библиотеку 

In [1]:
 pip install Flask-SQLAlchemy



   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
    --------------------------------------- 0.0/2.1 MB 217.9 kB/s eta 0:00:10
    --------------------------------------- 0.0/2.1 MB 217.9 kB/s eta 0:00:10
   - -------------------------------------- 0.1/2.1 MB 272.3 kB/s eta 0:00:08
   - -------------------------------------- 0.1/2.1 MB 280.5 kB/s eta 0:00:08
   - -------------------------------------- 0.1/2.1 MB 280.5 kB/s eta 0:00:08
   - -------------------------------------- 0.1/2.1 MB 294.4 kB/s eta 0:00:07
   -- ------------------------------------- 0.1/2.1 MB 327.4 kB/s eta 0:00:07
   --- ------------------------------------ 0.2/2.1 MB 403.5 kB/s eta 0:00:05
   --- ------------------------------------ 0.2/2.1 MB 429.5 kB/s eta 0:00:05
   ---- ----------------------------------- 0.2/2.1 MB 444.3 kB/s eta 0:00:05
   ---- ----


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from flask import Flask, render_template, request, redirect, url_for, flash
from flask_sqlalchemy import SQLAlchemy

app = Flask(__name__)
app.secret_key = 'Oni_ybily_Kenny'  # Создаем секретный ключ для работы с flash-сообщениями , у каждого должен быть индивидуальным ! 
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///users.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False

db = SQLAlchemy(app)

# Модель пользователя
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True) # Ваш id
    username = db.Column(db.String(80), nullable=False, unique=True) # Ваше имя 
    email = db.Column(db.String(120), nullable=False, unique=True) # Ваш email 
    password = db.Column(db.String(200), nullable=False) # Ваш пароль 

# Главная страница
@app.route('/')
def button_page():
    return render_template('button.html')

# Страница регистрации
@app.route('/register', methods=['GET', 'POST'])
def registration_page():
    if request.method == 'POST':
        username = request.form['username']
        email = request.form['email']
        password = request.form['password']

        # Проверка на существование пользователя
        existing_user = User.query.filter((User.username == username) | (User.email == email)).first()
        if existing_user:
            flash('Имя пользователя или email уже заняты', 'error')
            return redirect(url_for('registration_page'))

        # Добавление нового пользователя
        new_user = User(username=username, email=email, password=password)
        db.session.add(new_user)
        db.session.commit()

        flash('Регистрация прошла успешно!', 'success')
        return redirect(url_for('login_page'))

    return render_template('index_registration.html')

# Страница входа
@app.route('/login', methods=['GET', 'POST'])
def login_page():
    if request.method == 'POST':
        email = request.form['email']
        password = request.form['password']

        # Проверка учетных данных
        user = User.query.filter_by(email=email, password=password).first()
        if user:
            flash(f'Добро пожаловать, {user.username}!', 'success')
            return redirect(url_for('button_page'))
        else:
            flash('Неверный email или пароль', 'error')

    return render_template('index_login.html')

if __name__ == '__main__':
    with app.app_context():
        db.create_all()  # Создание таблиц в базе данных
    app.run(debug=True)


Теперь это стало больше стало похоже на полноценный сайт , осталось внести незначительные изменения в html код 


изменения в index_registration.html 


In [ ]:
<form class="registration-form" method="POST" action="/register">
    <label for="username">Имя пользователя</label>
    <input type="text" id="username" name="username" required>

    <label for="email">Электронная почта</label>
    <input type="email" id="email" name="email" required>

    <label for="password">Пароль</label>
    <input type="password" id="password" name="password" required>

    <button type="submit">Зарегистрироваться</button>
</form>


Создадим еще один файл index_login.html 

In [ ]:
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Вход</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='registration.css') }}">
</head>
<body>

<div class="registration-container">
    <h2>Вход</h2>
    <form class="registration-form" method="POST" action="/login">
        <label for="email">Электронная почта</label>
        <input type="email" id="email" name="email" required>

        <label for="password">Пароль</label>
        <input type="password" id="password" name="password" required>

        <button type="submit">Войти</button>
    </form>
</div>

</body>
</html>


In [ ]:
%shell
jupyter nbconvert --to html /content/lab_k_means.ipynb